In [1]:
from dotenv import load_dotenv

from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_core.tools import tool

from langchain_google_genai import ChatGoogleGenerativeAI

In [2]:
load_dotenv()

True

In [3]:
search = GoogleSerperAPIWrapper(type = 'search')

In [4]:
# Step 1: Creating tools

@tool
def google_search(query:str) ->any:
    '''
    This function will take a query as input and search for the answer on Google Search Engine.
    Pass any query to it and it will return results by searching on Google Search Engine
    '''

    return search.results(query)

#--------------------------------------------------------------------------------------------------


import requests
@tool
def whether_search(city:str) ->str:
    '''
    This function returns the live whether data of any city in the world.
    Pass any city name and it will return all the whether details.
    '''

    api_key = '6018a20db02a3eb6aa1ef7bffa662350'

    url = f"http://api.weatherstack.com/current?access_key={api_key}&query={city}"

    response = requests.get(url)

    return response.json()

#--------------------------------------------------------------------------------------------------

@tool
def fetch_train_details(train_number: str) -> str:

    '''
    Given train number, this function fetches all the deetails regarding
    train like train schedule, stations, station timings, when train reaches at different stations,
    Time when train departs from in-between stations, Names of In-between stations. 
    '''
    url = "https://irctc-insight.p.rapidapi.com/api/v1/train-details"
    headers = {
        "x-rapidapi-key": "a386df66c3msha0ef942c346306dp107a95jsn1c48df211ae1",
        "x-rapidapi-host": "irctc-insight.p.rapidapi.com",
        "Content-Type": "application/json"
    }
    payload = {"trainNo": train_number}
    
    response = requests.post(url, json = payload, headers = headers)
    return response.json()

In [5]:
whether_search.invoke({'city': 'Rajkot'})

{'request': {'type': 'City',
  'query': 'Rajkot, India',
  'language': 'en',
  'unit': 'm'},
 'location': {'name': 'Rajkot',
  'country': 'India',
  'region': 'Gujarat',
  'lat': '22.300',
  'lon': '70.783',
  'timezone_id': 'Asia/Kolkata',
  'localtime': '2025-09-30 22:29',
  'localtime_epoch': 1759271340,
  'utc_offset': '5.50'},
 'current': {'observation_time': '04:59 PM',
  'temperature': 25,
  'weather_code': 176,
  'weather_icons': ['https://cdn.worldweatheronline.com/images/wsymbols01_png_64/wsymbol_0025_light_rain_showers_night.png'],
  'weather_descriptions': ['Patchy rain nearby'],
  'astro': {'sunrise': '06:38 AM',
   'sunset': '06:35 PM',
   'moonrise': '01:52 PM',
   'moonset': 'No moonset',
   'moon_phase': 'Waxing Gibbous',
   'moon_illumination': 50},
  'air_quality': {'co': '162.85',
   'no2': '5.65',
   'o3': '72',
   'so2': '1.85',
   'pm2_5': '4.65',
   'pm10': '5.05',
   'us-epa-index': '1',
   'gb-defra-index': '1'},
  'wind_speed': 27,
  'wind_degree': 175,
  'wi

In [6]:
# Step 2: Creating LLM

llm = ChatGoogleGenerativeAI(
    model = "gemini-2.5-flash"
)

In [7]:
# Step 3: Creting functional_agent and agent executor

from langchain.agents import create_openai_functions_agent, AgentExecutor
from langchain import hub

# creating prompt
prompt = hub.pull("hwchase17/openai-functions-agent")

# creating functional_agent
functional_agent = create_openai_functions_agent(
    llm = llm,
    tools = [google_search, whether_search, fetch_train_details],
    prompt = prompt
)

# creating agent executor

agent_executor = AgentExecutor(
    agent = functional_agent,
    tools = [google_search, whether_search, fetch_train_details],
    verbose = True,
    handle_parsing_errors = True
)

c:\Users\jugal\langchain-env\Lib\site-packages\langsmith\client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [8]:
# Step 5: Invoke

response = agent_executor.invoke({"input": "When train number 22959 arrives at Rajtot Junction?"})



> Entering new AgentExecutor chain...

Invoking: `fetch_train_details` with `{'train_number': '22959'}`


{'trainNumber': '22959', 'trainName': 'INTERCITY EXP', 'route': 'Vadodara Junction to Jamnagar', 'runningDays': {'Mon': True, 'Tue': True, 'Wed': True, 'Thu': True, 'Fri': True, 'Sat': True, 'Sun': True}, 'trainRoute': [{'sno': '1', 'stationName': 'Vadodara Jn - BRC', 'arrives': 'Start', 'departs': '15:50', 'haltTime': '-', 'distance': '0.0 km', 'avgDelay': '01 Min', 'day': '1'}, {'sno': '2', 'stationName': 'Anand Jn - ANND', 'arrives': '16:11', 'departs': '16:13', 'haltTime': '2m', 'distance': '35.5 km', 'avgDelay': '14 Min', 'day': '1'}, {'sno': '3', 'stationName': 'Nadiad Jn - ND', 'arrives': '16:27', 'departs': '16:29', 'haltTime': '2m', 'distance': '54.2 km', 'avgDelay': '19 Min', 'day': '1'}, {'sno': '4', 'stationName': 'Ahmedabad Jn - ADI', 'arrives': '17:30', 'departs': '17:35', 'haltTime': '5m', 'distance': '100.0 km', 'avgDelay': '13 Min', 'day': '1'}, {'sno': '5', 'sta

In [9]:
print(response['input'])
print()

print(response['output'])

When train number 22959 arrives at Rajtot Junction?

Train number 22959 arrives at Rajkot Junction at 21:44.


In [10]:
# Step 5: Invoke

response2 = agent_executor.invoke({
    "input": '''
    Find the current whether condition of the capital of Gujarat, India'''})



> Entering new AgentExecutor chain...

Invoking: `google_search` with `{'query': 'capital of Gujarat, India'}`


{'searchParameters': {'q': 'capital of Gujarat, India', 'gl': 'us', 'hl': 'en', 'type': 'search', 'num': 10, 'engine': 'google'}, 'organic': [{'title': 'Gandhinagar', 'link': 'https://en.wikipedia.org/wiki/Gandhinagar', 'snippet': 'Gandhinagar is the capital of the state of Gujarat in India. Gandhinagar is located approximately 23 kilometres (14 mi) north of Ahmedabad, on the west ...', 'position': 1}, {'title': 'Gujarat', 'link': 'https://en.wikipedia.org/wiki/Gujarat', 'snippet': "Gujarat's capital city is Gandhinagar, while its largest city is Ahmedabad. The Gujaratis are indigenous to the state and their language, Gujarati, is the ...", 'position': 2}, {'title': 'Gandhinagar | Map, History, & Facts', 'link': 'https://www.britannica.com/place/Gandhinagar', 'snippet': 'Gandhinagar, city, capital of Gujarat state, west-central India. It lies on the banks of the Sabarmati 

In [11]:
print(response2['output'])

The current weather in Gandhinagar, Gujarat, India is 24°C with Mist. The wind speed is 15 km/h in the SSE direction. Humidity is at 83% and visibility is 4 km. The UV Index is 0.


In [12]:
# See, it first invoked Google_Search to find the capita of Gujarat and then 
# invoked whether_search and fetched current whether details.

In [ ]:
import requests

@tool
def fetch_train_details(train_number: str) -> str:
    url = "https://irctc-insight.p.rapidapi.com/api/v1/train-details"
    headers = {
        "x-rapidapi-key": "a386df66c3msha0ef942c346306dp107a95jsn1c48df211ae1",
        "x-rapidapi-host": "irctc-insight.p.rapidapi.com",
        "Content-Type": "application/json"
    }
    payload = {"trainNo": train_number}
    
    response = requests.post(url, json = payload, headers = headers)
    print(response.json())


ValueError: Function must have a docstring if description not provided.